# Import Packages and Libraries

In [ ]:
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

from sklearn.decomposition import PCA

from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold


# Importing Data

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Tabular Playground October 2022/Data/train_0.csv')
#for i in range(1, 2):
  #temp_df = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Tabular Playground October 2022/Data/train_1.csv')
  #df = df.append(temp_df)
  #print('df',i,'has been merged')

df = df.copy()#[:100000] # The More Data Read, The Higher the Score

### Helper Methods

In [ ]:
def distance_to(df, name_1, name_2):

  name_1_coor = np.array([df[name_1+'_pos_x'], df[name_1+'_pos_y'], df[name_1+'_pos_z']]).T
  name_2_coor = np.array([df[name_2+'_pos_x'], df[name_2+'_pos_y'], df[name_2+'_pos_z']]).T
  distances = []
  for i in range(name_1_coor.shape[0]):
    distances.append(np.linalg.norm(name_2_coor[i] - name_1_coor[i]))
  return distances

def distance_to_goal(df, name_1, goal_x, goal_y, goal_z):

  name_1_coor = np.array([df[name_1+'_pos_x'], df[name_1+'_pos_y'], df[name_1+'_pos_z']]).T
  name_2_coor = np.array([[goal_x]*df.shape[0], [goal_y]*df.shape[0], [goal_z]*df.shape[0]]).T
  distances = []
  for i in range(name_1_coor.shape[0]):
    distances.append(np.linalg.norm(name_2_coor[i] - name_1_coor[i]))
  return distances

# Data Preproxessing

### Encoding Data

In [ ]:
def encode_team_scoring_next(df):
  df.loc[df['team_scoring_next'].isna(), 'team_scoring_next'] = 0 # Fix NA Values

  df.loc[df['team_scoring_next']=='A', 'team_scoring_next'] = 1 # Encode Team A
  df.loc[df['team_scoring_next']=='B', 'team_scoring_next'] = 2 # Encode Team B
  return df

### Data Imputation

In [ ]:
def fix_na_demolished(X):
  for player in ['p0','p1','p2','p3','p4','p5']:
    X.loc[X[player+'_boost'].isna(), player+'_is_demolished'] = 1
    X.loc[~X[player+'_boost'].isna(), player+'_is_demolished'] = 0

    X.loc[X[player+'_pos_y'].isna(), player+'_pos_y'] = 0
    X.loc[X[player+'_pos_x'].isna(), player+'_pos_x'] = 0
    X.loc[X[player+'_pos_z'].isna(), player+'_pos_z'] = 0
    X.loc[X[player+'_vel_y'].isna(), player+'_vel_y'] = 0
    X.loc[X[player+'_vel_x'].isna(), player+'_vel_x'] = 0
    X.loc[X[player+'_vel_z'].isna(), player+'_vel_z'] = 0
    X.loc[X[player+'_boost'].isna(), player+'_boost'] = 0
  return X

### Reducing Dimentionality of Data using PCA

In [ ]:
# We did it manually, can be done using a for loopas with fix_na_demolished
def apply_pca(X):
  pca = PCA(n_components = 1)

  ball_pos = pca.fit_transform(X[['ball_pos_x','ball_pos_y','ball_pos_z']]).tolist()
  ball_vel = pca.fit_transform(X[['ball_vel_x','ball_vel_y','ball_vel_z']]).tolist()

  playerA_1_pos = pca.fit_transform(X[['p0_pos_x','p0_pos_y','p0_pos_z','p0_is_demolished']]).tolist()
  playerA_1_vel = pca.fit_transform(X[['p0_vel_x','p0_vel_y','p0_vel_z','p0_is_demolished']]).tolist()
  playerA_1_boost = pca.fit_transform(X[['p0_boost','boost0_timer','p0_is_demolished']]).tolist()

  playerA_2_pos = pca.fit_transform(X[['p1_pos_x','p1_pos_y','p1_pos_z','p1_is_demolished']]).tolist()
  playerA_2_vel = pca.fit_transform(X[['p1_vel_x','p1_vel_y','p1_vel_z','p1_is_demolished']]).tolist()
  playerA_2_boost = pca.fit_transform(X[['p1_boost','boost1_timer','p1_is_demolished']]).tolist()

  playerA_3_pos = pca.fit_transform(X[['p2_pos_x','p2_pos_y','p2_pos_z','p2_is_demolished']]).tolist()
  playerA_3_vel = pca.fit_transform(X[['p2_vel_x','p2_vel_y','p2_vel_z','p2_is_demolished']]).tolist()
  playerA_3_boost = pca.fit_transform(X[['p2_boost','boost2_timer','p2_is_demolished']]).tolist()

  playerB_1_pos = pca.fit_transform(X[['p3_pos_x','p3_pos_y','p3_pos_z','p3_is_demolished']]).tolist()
  playerB_1_vel = pca.fit_transform(X[['p3_vel_x','p3_vel_y','p3_vel_z','p3_is_demolished']]).tolist()
  playerB_1_boost = pca.fit_transform(X[['p3_boost','boost3_timer','p3_is_demolished']]).tolist()

  playerB_2_pos = pca.fit_transform(X[['p4_pos_x','p4_pos_y','p4_pos_z','p4_is_demolished']]).tolist()
  playerB_2_vel = pca.fit_transform(X[['p4_vel_x','p4_vel_y','p4_vel_z','p4_is_demolished']]).tolist()
  playerB_2_boost = pca.fit_transform(X[['p4_boost','boost4_timer','p4_is_demolished']]).tolist()

  playerB_3_pos = pca.fit_transform(X[['p5_pos_x','p5_pos_y','p5_pos_z','p5_is_demolished']]).tolist()
  playerB_3_vel = pca.fit_transform(X[['p5_vel_x','p5_vel_y','p5_vel_z','p5_is_demolished']]).tolist()
  playerB_3_boost = pca.fit_transform(X[['p5_boost','boost5_timer','p5_is_demolished']]).tolist()

  train_X = pd.DataFrame()
  train_X['ball_pos'] = np.array(ball_pos).reshape(1,-1)[0]
  train_X['ball_vel'] = np.array(ball_vel).reshape(1,-1)[0]

  train_X['playerA_1_pos'] = np.array(playerA_1_pos).reshape(1,-1)[0]
  train_X['playerA_1_vel'] = np.array(playerA_1_vel).reshape(1,-1)[0]
  train_X['playerA_1_boost'] = np.array(playerA_1_boost).reshape(1,-1)[0]

  train_X['playerA_2_pos'] = np.array(playerA_2_pos).reshape(1,-1)[0]
  train_X['playerA_2_vel'] = np.array(playerA_2_vel).reshape(1,-1)[0]
  train_X['playerA_2_boost'] = np.array(playerA_2_boost).reshape(1,-1)[0]

  train_X['playerA_3_pos'] = np.array(playerA_3_pos).reshape(1,-1)[0]
  train_X['playerA_3_vel'] = np.array(playerA_3_vel).reshape(1,-1)[0]
  train_X['playerA_3_boost'] = np.array(playerA_3_boost).reshape(1,-1)[0]

  train_X['playerB_1_pos'] = np.array(playerB_1_pos).reshape(1,-1)[0]
  train_X['playerB_1_vel'] = np.array(playerB_1_vel).reshape(1,-1)[0]
  train_X['playerB_1_boost'] = np.array(playerB_1_boost).reshape(1,-1)[0]

  train_X['playerB_2_pos'] = np.array(playerB_2_pos).reshape(1,-1)[0]
  train_X['playerB_2_vel'] = np.array(playerB_2_vel).reshape(1,-1)[0]
  train_X['playerB_2_boost'] = np.array(playerB_2_boost).reshape(1,-1)[0]

  train_X['playerB_3_pos'] = np.array(playerB_3_pos).reshape(1, -1)[0]
  train_X['playerB_3_vel'] = np.array(playerB_3_vel).reshape(1, -1)[0]
  train_X['playerB_3_boost'] = np.array(playerB_3_boost).reshape(1, -1)[0]

  return train_X

# Feature Engineering

Adding Features:
  - Distance of each player to ball, goalA, goalB
  - Distance of each team to ball, goalA, goalB
  - Team with player closest to ball
  - Team closest to ball


In [ ]:
def add_new_features(train_X, X):
  players = ['p0','p1','p2','p3','p4','p5']
  new_name = ['playerA_1','playerA_2','playerA_3','playerB_1','playerB_2','playerB_3']

  for player, name in zip(players, new_name):
    train_X[name+'_to_ball'] = distance_to(X, player, 'ball')
    train_X[name+'_to_goalA'] = distance_to_goal(X, player, 0, -104.3092, 20.439399)
    train_X[name+'_to_goalB'] = distance_to_goal(X, player, 0, 104.3092, 20.439399)
  
  train_X['teamA_to_ball'] = train_X[['playerA_1_to_ball','playerA_2_to_ball','playerA_3_to_ball']].mean(axis=1)
  train_X['teamB_to_ball'] = train_X[['playerB_1_to_ball','playerB_2_to_ball','playerB_3_to_ball']].mean(axis=1)

  train_X['teamA_to_goalA'] = train_X[['playerA_1_to_goalA','playerA_2_to_goalA','playerA_3_to_goalA']].mean(axis=1)
  train_X['teamB_to_goalA'] = train_X[['playerB_1_to_goalA','playerB_2_to_goalA','playerB_3_to_goalA']].mean(axis=1)

  train_X['teamA_to_goalB'] = train_X[['playerA_1_to_goalB','playerA_2_to_goalB','playerA_3_to_goalB']].mean(axis=1)
  train_X['teamB_to_goalB'] = train_X[['playerB_1_to_goalB','playerB_2_to_goalB','playerB_3_to_goalB']].mean(axis=1)

  teamA_ball = ['playerA_1_to_ball','playerA_2_to_ball','playerA_3_to_ball']
  teamB_ball = ['playerB_1_to_ball','playerB_2_to_ball','playerB_3_to_ball']

  # Team with Player Closest to Ball
  train_X.loc[train_X[teamA_ball].min(axis=1) < train_X[teamB_ball].min(axis=1), 'player_team_closest_to_ball'] = 1
  train_X.loc[train_X[teamA_ball].min(axis=1) > train_X[teamB_ball].min(axis=1), 'player_team_closest_to_ball'] = 2
  train_X.loc[train_X[teamA_ball].min(axis=1) == train_X[teamB_ball].min(axis=1), 'player_team_closest_to_ball'] = 0

  # Team closest to Ball
  train_X.loc[train_X['teamA_to_ball'] < train_X['teamB_to_ball'], 'team_closest_to_ball'] = 1
  train_X.loc[train_X['teamA_to_ball'] > train_X['teamB_to_ball'], 'team_closest_to_ball'] = 2
  train_X.loc[train_X['teamA_to_ball'] == train_X['teamB_to_ball'], 'team_closest_to_ball'] = 0
  
  return train_X

 # Putting it all together - Predictions

In [ ]:
df = encode_team_scoring_next(df)

df = fix_na_demolished(df)
train_df = apply_pca(df)
X = add_new_features(train_df, df)

A = df['team_A_scoring_within_10sec']
B = df['team_B_scoring_within_10sec']


Creating + Training + Evaluating A Model (XGboost)

In [ ]:
def evaluate_model(model, X, y, X_final_test):
  scores = []
  cv = StratifiedKFold(n_splits=2, shuffle=True)

  for fold, (train_idx, test_idx) in enumerate(cv.split(X, y)):
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

    model.fit(X_train, y_train) # Training Model

    predictions = model.predict_proba(X_test)[:, 1] # Predicting Model, but probaility of being 1 instead of 0
    score = roc_auc_score(y_test, predictions)
    scores.append(score)
    print(f"Fold {fold + 1} \t\t AUC: {score}")

  return model.predict_proba(X_final_test)[:, 1]

def predict(model, X, y, X_test):
  model.fit(X, y)
  return model.predict_proba(X_test)[:, 1]


# Hyperparameter Tuning

In [ ]:
from xgboost import XGBClassifier

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()


params = {'n_estimators': 200,
          'max_depth': 7,
          'learning_rate': 0.1,
          'objective': 'binary:logistic'}

model = XGBClassifier(**params)

#teamA_pred = evaluate_model(model, scaler.fit_transform(X), A, scaler.fit_transform(X))
#teamB_pred = evaluate_model(model, scaler.fit_transform(X), B, scaler.fit_transform(X))

# Final Submisson

In [ ]:
test_X = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Tabular Playground October 2022/test.csv')
test_X.head()

In [ ]:
test_X = fix_na_demolished(test_X)
final_X = apply_pca(test_X)
final_X = add_new_features(final_X, test_X)

final_X = pd.read_csv('/content/drive/MyDrive/Kaggle Competitions/Tabular Playground October 2022/X_test.csv').drop('Unnamed: 0', axis=1)
final_X.shape

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X = scaler.fit_transform(X)
final_X = scaler.fit_transform(final_X)

params = {'num_boost_round': 10,
          'max_depth': 20}
          #'early_stopping_rounds': 15}
          #'n_estimators': 500,
          #'early_stopping_rounds': 15,
          #'learning_rate': 0.1,
          #'objective': 'binary:logistic'}

model = XGBClassifier(**params)

predA = predict(model, X, A, final_X)
predB = predict(model, X, B, final_X)


In [ ]:
submission = test_X.copy()[['id']]
submission['team_A_scoring_within_10sec'] = predA
submission['team_B_scoring_within_10sec'] = predB

submission

In [ ]:
submission.to_csv('/content/drive/MyDrive/Kaggle Competitions/Tabular Playground October 2022/submission.csv', index=False)